<a href="https://colab.research.google.com/github/shubham-senani/Marketing-Analytics/blob/main/MA16_MMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marketing Mix Modeling

### Libraries & Data

In [ ]:
# ==========================================
# INSTALL REQUIRED LIBRARIES (Colab only)
# ==========================================


# ==========================================
# IMPORT LIBRARIES
# ==========================================
import numpy as np
import pandas as pd

import re
import io
from google.colab import files

import statsmodels.formula.api as smf
from scipy.optimize import minimize

In [ ]:
def load_dataset(google_sheet_id=None):
    """
    Loads dataset either by:
    1 → Uploading a CSV file manually
    2 → Loading a Google Sheet (if ID is provided)

    Parameters:
    google_sheet_id (str, optional): Google Sheet ID.
                                     If provided, automatically loads Sheet.

    Returns:
    pd.DataFrame
    """

    # -----------------------------------
    # IF GOOGLE SHEET ID IS PROVIDED → AUTO LOAD (Option 2)
    # -----------------------------------
    if google_sheet_id is not None:
        data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
        df = pd.read_csv(data_url)
        print("Google Sheet loaded successfully.")

    else:
        # Ask user only if ID not provided
        print("Choose how you want to provide the dataset:")
        print("1 → Upload CSV file (from your computer)")
        print("2 → Load Google Sheet (as CSV)")

        choice = input("Enter 1 or 2: ").strip()

        # OPTION 1: Upload CSV
        if choice == "1":
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
            print(f"File '{filename}' uploaded successfully.")

        # OPTION 2: Ask for Google Sheet ID
        elif choice == "2":
            google_sheet_id = input("Enter the Google Sheet ID: ").strip()
            data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
            df = pd.read_csv(data_url)
            print("Google Sheet loaded successfully.")

        else:
            raise ValueError("Invalid choice. Please rerun and enter 1 or 2.")

    # -----------------------------------
    # DATA CHECK
    # -----------------------------------
    print("\nDataset shape:", df.shape)
    print("Columns:", df.columns.tolist())

    return df

## Non-Linear Optimization

## Regression

In [ ]:
# Call the custom function to load the dataset
# The function will prompt you to either:
# 1) Upload a CSV file manually, or
# 2) Enter a Google Sheet ID to load data directly - 18h3FKG3tYzlFSuD3E38ZNKNANzOAtndT

df = load_dataset(google_sheet_id="18h3FKG3tYzlFSuD3E38ZNKNANzOAtndT")

# Display the first 5 rows of the dataset
df.head()

Google Sheet loaded successfully.

Dataset shape: (36, 6)
Columns: ['year', 'Month', 'base_price', 'Promo_AdjustedPrice', 'TV_Advertising', 'Volume']


,year,Month,base_price,Promo_AdjustedPrice,TV_Advertising,Volume
0,2016,1,40,38.0,200000000,4012548.146
1,2016,2,40,36.8,150000000,3943150.589
2,2016,3,40,36.8,0,2225250.681
3,2016,4,40,36.0,0,2048853.701
4,2016,5,40,36.8,0,1906784.287


**Use Case**

A mosquito repellent brand wants to allocate its **monthly marketing budget efficiently** between two channels:

1. **TV advertising**
2. **Price promotion**



**Historical Data**: The dataset contains **3 years of past demand data**, including:

- Monthly **sales volume**
- **TV advertising spend**
- **Promotion-adjusted price**

In [ ]:
demand_data = df.copy()

# -----------------------------
# PREPARE DATA
# -----------------------------

# Make Month categorical for seasonal dummy creation
demand_data["Month"] = demand_data["Month"].astype("category")

# -----------------------------
# LOG TRANSFORM VARIABLES
# -----------------------------
# Multiplicative demand model:
# Q = A * TV_Advertising^(adv_elast) * Price^(price_elast)
#
# Taking logs:
# log(Q) = intercept + month_effects + adv_elast*log(TV) + price_elast*log(Price)
#
# Adding +1 to TV advertising avoids log(0) errors if spend is zero.
demand_data["log_AdStock"] = np.log(demand_data["TV_Advertising"] + 1)
demand_data["log_price"] = np.log(demand_data["Promo_AdjustedPrice"])
demand_data["log_q"] = np.log(demand_data["Volume"])

demand_data.head()

,year,Month,base_price,Promo_AdjustedPrice,TV_Advertising,Volume,log_AdStock,log_price,log_q
0,2016,1,40,38.0,200000000,4012548.146,19.113828,3.637586,15.204937
1,2016,2,40,36.8,150000000,3943150.589,18.826146,3.605498,15.187491
2,2016,3,40,36.8,0,2225250.681,0.000000,3.605498,14.615380
3,2016,4,40,36.0,0,2048853.701,0.000000,3.583519,14.532791
4,2016,5,40,36.8,0,1906784.287,0.000000,3.605498,14.460929


**Modeling Approach**

Demand is modeled using a **multiplicative form**.

To make estimation easier:

- We **take the logarithm of the demand function**
- This converts it into a **linear regression model**

To capture **seasonality**, the **Month variable is included as a categorical feature**.

In [ ]:
# -----------------------------
# REGRESSION MODEL
# -----------------------------
# Month is included as seasonality control.
# In dummy coding, one month is omitted automatically as baseline.
# If Month1 / January is baseline, negative month coefficients mean
# those months have lower demand than January.

model = smf.ols("log_q ~ C(Month) + log_price + log_AdStock", data=demand_data).fit()

print("=" * 70)
print("REGRESSION SUMMARY")
print("=" * 70)
print(model.summary())

REGRESSION SUMMARY
                            OLS Regression Results                            
Dep. Variable:                  log_q   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                     7985.
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           2.88e-37
Time:                        06:37:29   Log-Likelihood:                 146.09
No. Observations:                  36   AIC:                            -264.2
Df Residuals:                      22   BIC:                            -242.0
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         18.5497

**Business Insights from the Case**

The expected economic relationships are:

- **Price elasticity** → Expected to be **negative**  
- **TV advertising elasticity** → Expected to be **positive**  
- **Month dummy variables** → Capture **seasonal demand variation**

Month is treated as a categorical variable so **January becomes the baseline month**.

In [ ]:
# -----------------------------
# EXTRACT KEY COEFFICIENTS
# -----------------------------
# These are the most important parameters for marketing decisions:
# - price elasticity
# - advertising elasticity
price_elast = model.params["log_price"]
adv_elast = model.params["log_AdStock"]
intercept = model.params["Intercept"]

print("\n" + "=" * 70)
print("KEY ELASTICITIES")
print("=" * 70)
print(f"Price elasticity estimate       : {price_elast:.6f}")
print(f"Advertising elasticity estimate : {adv_elast:.6f}")


KEY ELASTICITIES
Price elasticity estimate       : -1.039152
Advertising elasticity estimate : 0.022740


In [ ]:
# -----------------------------
# HISTORICAL AGGREGATES
# -----------------------------
# The dataset columns:
# year, Month, base_price, Promo_AdjustedPrice, TV_Advertising, Volume
#
# So the correct full/base price column is: base_price

base_price_col = "base_price"

total_revenue_crore = (
    demand_data["Volume"] * demand_data["Promo_AdjustedPrice"]
).sum() / 1e7

total_tv_cost_crore = demand_data["TV_Advertising"].sum() / 1e7

total_promo_cost_crore = (
    demand_data["Volume"] * (demand_data[base_price_col] - demand_data["Promo_AdjustedPrice"])
).sum() / 1e7

print("\n" + "=" * 70)
print("HISTORICAL AGGREGATES (IN ₹ CRORE)")
print("=" * 70)
print(f"Total revenue              : {total_revenue_crore:.4f}")
print(f"Total TV advertising cost  : {total_tv_cost_crore:.4f}")
print(f"Total promotion cost       : {total_promo_cost_crore:.4f}")


HISTORICAL AGGREGATES (IN ₹ CRORE)
Total revenue              : 406.8096
Total TV advertising cost  : 343.0000
Total promotion cost       : 26.8396


## Optimization

**Objective**

For the **next year**, the brand has a **monthly marketing budget of ₹10 crore**.

The optimization is done **for one month only** (January baseline).

Goal

1. Maximize **Revenue**: Revenue = Quantity * Price

2. Constraint: TV Spend + Cost of Price Promotion <= Budget

**Important Note**

* Most optimization solvers **minimize by default**.

* Therefore, we **minimize the negative of revenue** to effectively **maximize revenue**.

In [ ]:
# -----------------------------
# OPTIMIZATION SETUP
# -----------------------------
# We optimize for one month only.
# Since the original case solves for the first month (baseline month),
# month dummy effects are omitted in the optimization formula.
#
# Decision variables:
# x[0] = TV advertising spend for the month
# x[1] = promotional selling price for the month
#
# Budget values are in rupees.
# 1 crore = 10^7 rupees

budget = 10 * 10**7    # ₹10 crore monthly budget
full_price = 49.0      # given in the case for next-year optimization
min_price = 40.0       # lower guardrail for promotional price

# NOTE:
# In regression, advertising entered as:
# log_AdStock = log(TV_Advertising + 1)
#
# To remain consistent with the estimated model, use log(tv_spend + 1)
# in optimization as well.

def predicted_quantity(tv_spend, promo_price):
    """
    Predict quantity using the estimated log-log demand model
    for the baseline month (January / omitted month dummy).
    """
    tv_spend = max(tv_spend, 0.0)
    promo_price = max(promo_price, 1e-8)

    log_q_hat = (
        intercept
        + adv_elast * np.log(tv_spend + 1)
        + price_elast * np.log(promo_price)
    )
    return np.exp(log_q_hat)


def objective(x):
    """
    Objective for optimization:
    maximize revenue = quantity * price

    Since scipy.minimize() minimizes by default,
    return negative revenue.
    """
    tv_spend, promo_price = x
    quantity = predicted_quantity(tv_spend, promo_price)
    revenue = quantity * promo_price
    return -revenue


def budget_constraint(x):
    """
    Nonlinear budget constraint:

    TV spend + promotion cost <= budget

    Promotion cost =
    quantity * (full_price - promo_price)

    scipy inequality format requires:
    g(x) >= 0

    So we write:
    budget - [TV spend + promotion cost] >= 0
    """
    tv_spend, promo_price = x
    quantity = predicted_quantity(tv_spend, promo_price)
    promo_cost = quantity * (full_price - promo_price)
    return budget - (tv_spend + promo_cost)


In [ ]:
# Initial guess from the R example
x0 = np.array([7 * 10**7, 48.0])   # ₹7 crore TV spend, price ₹48

# Bounds:
# TV spend: 0 to ₹10 crore
# Promo price: ₹40 to ₹49
bounds = [
    (0, budget),
    (min_price, full_price)
]

constraints = [
    {"type": "ineq", "fun": budget_constraint}
]

In [ ]:
# -----------------------------
# RUN NONLINEAR OPTIMIZATION
# -----------------------------
result = minimize(
    fun=objective,
    x0=x0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-9, "disp": True, "maxiter": 1000}
)

Optimization terminated successfully    (Exit mode 0)
            Current function value: -148377959.0227795
            Iterations: 24
            Function evaluations: 142
            Gradient evaluations: 21


In [ ]:
# -----------------------------
# PRINT OPTIMIZATION RESULTS
# -----------------------------
print("\n" + "=" * 70)
print("OPTIMIZATION RESULTS")
print("=" * 70)

if result.success:
    opt_tv_spend = result.x[0]
    opt_price = result.x[1]

    opt_quantity = predicted_quantity(opt_tv_spend, opt_price)
    opt_revenue = opt_quantity * opt_price
    opt_promo_cost = opt_quantity * (full_price - opt_price)
    total_cost = opt_tv_spend + opt_promo_cost

    print(f"Optimization successful: {result.success}")
    print(f"Solver message         : {result.message}")
    print(f"Optimal TV spend       : ₹{opt_tv_spend / 1e7:.4f} crore")
    print(f"Optimal promo price    : ₹{opt_price:.5f}")
    print(f"Optimal quantity       : {opt_quantity:,.0f} units")
    print(f"Optimal revenue        : ₹{opt_revenue / 1e7:.4f} crore")
    print(f"Promo cost implied     : ₹{opt_promo_cost / 1e7:.4f} crore")
    print(f"Total spend used       : ₹{total_cost / 1e7:.4f} crore")
    print(f"Unused budget          : ₹{(budget - total_cost) / 1e7:.6f} crore")
else:
    print("Optimization failed.")
    print(result.message)


OPTIMIZATION RESULTS
Optimization successful: True
Solver message         : Optimization terminated successfully
Optimal TV spend       : ₹7.0000 crore
Optimal promo price    : ₹40.75907
Optimal quantity       : 3,640,367 units
Optimal revenue        : ₹14.8378 crore
Promo cost implied     : ₹3.0000 crore
Total spend used       : ₹10.0000 crore
Unused budget          : ₹0.000000 crore


**Important Interpretation Notes**

1. Price Elasticity
* If price elasticity is around **-1.04**, then a **1% increase in price**
  reduces quantity demanded by about **1.04%**, holding other factors constant.

2. Advertising Elasticity
* If advertising elasticity is around **0.023**, then a **1% increase in TV spend**
  increases quantity demanded by about **0.023%**, holding other factors constant.

3. Monthly Seasonality
* In the regression, **Month is included as a categorical variable**.
* Significant **negative month coefficients** mean those months have **lower demand**
  than the **omitted baseline month**.

4. Optimization Trade-off
The solver balances two key decisions:

* Spending more on **TV advertising** to stimulate demand
* **Reducing price** to stimulate demand while accounting for **promotion cost**

5. Business Relevance
This example shows a **one-product, one-month optimization illustration**.

In practice, **FMCG / CPG firms** typically optimize across:

* Multiple **SKUs and brands**
* **Cannibalization effects** between products
* **Portfolio revenue and profit**
* **Spend guardrails** to avoid unrealistic changes